# Notebook 01: Dataset Exploration — From Raw GeoJSON to Clean IR Schema

## KLEAN Chatbot — Restaurant Information Retrieval & Recommendation

### Overview & Workflow
In a real-world Information Retrieval system, raw geospatial data often contains irregular formatting, empty strings, stringified lists, and missing identifiers.

This notebook demonstrates the complete **End-to-End Data Ingestion & Exploration pipeline**:
1. **Raw GeoJSON Inspection**: Inspect `places-phnom-penh-casual-local-eats-hrfrvcgb.geojson` as received.
2. **Data Issues & Anomalies**: Identify 6 real-world format quirks that prevent direct IR indexing.
3. **Data Cleaning & Schema Normalization**: Walk through the transformation from raw GeoJSON feature to clean standardized restaurant record.
4. **Before-and-After Comparison**: Compare a raw GeoJSON record directly with its cleaned representation.
5. **Clean Data Profiling**: Analyze attribute completeness, category distributions, bilingual language mix, and spatial coordinates across Phnom Penh.


In [1]:
import sys
import json
import re
from pathlib import Path
import pandas as pd

# Add project root to sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.data_loader import DataLoader, clean_category_tag

print(f"Project root: {PROJECT_ROOT}")


Project root: D:\Year4\Web_Retrive\Project


---
## 1. Inspect Raw GeoJSON (`places-phnom-penh-casual-local-eats-hrfrvcgb.geojson`)

Let's load the raw GeoJSON file directly without any preprocessing and inspect its internal structure.


In [2]:
raw_geojson_path = PROJECT_ROOT / "dataset" / "places-phnom-penh-casual-local-eats-hrfrvcgb.geojson"

with open(raw_geojson_path, "r", encoding="utf-8") as f:
    raw_geojson = json.load(f)

print(f"GeoJSON Top-level Keys: {list(raw_geojson.keys())}")
print(f"Total Features (Places) in GeoJSON: {len(raw_geojson['features'])}")

# Inspect first raw feature
sample_feature = raw_geojson["features"][0]
print("\nSample Feature #1 (Raw):")
print(json.dumps(sample_feature, indent=2, ensure_ascii=False))


GeoJSON Top-level Keys: ['type', 'features']
Total Features (Places) in GeoJSON: 698

Sample Feature #1 (Raw):
{
  "type": "Feature",
  "properties": {
    "name": "13Eo Breakfast",
    "brand_name": "",
    "street_address": "13Eo Breakfast",
    "city": "ភ្នំពេញ",
    "state": "",
    "postal_code": "121201",
    "country": "KH",
    "primary_category": "breakfast_and_brunch_restaurant",
    "sub_categories": "[\"vietnamese_restaurant\",\"tea_room\"]",
    "website": "",
    "phone": "+85511256604",
    "email": "",
    "social": "https://www.facebook.com/482898885219887",
    "longitude": 104.93585205078125,
    "latitude": 11.53061580657959
  },
  "geometry": {
    "type": "Point",
    "coordinates": [
      104.93585205078125,
      11.53061580657959
    ]
  }
}


---
## 2. Diagnosing Raw Data Issues & Quirks

By inspecting the raw GeoJSON feature above, we can identify **6 concrete issues** that make it unusable for Information Retrieval without cleaning:

| # | Raw Attribute | Issue in Raw GeoJSON | Why It Breaks IR / RAG | Cleaned Target |
|---|---|---|---|---|
| **1** | `feature` ID | No numeric integer `id` attribute exists. | Inverted index postings and FAISS matrix offsets require integer IDs (`doc_id`). | Assign integer ID `1, 2, ...` |
| **2** | `sub_categories` | Serialized JSON string: `'["vietnamese_restaurant","tea_room"]'` | Cannot filter or tokenize as a Python list; quotes and brackets pollute text. | True Python list `['Vietnamese Restaurant', 'Tea Room']` |
| **3** | `primary_category` | Snake_case tag: `'breakfast_and_brunch_restaurant'` | Low readability in search snippets and LLM prompts. | Title case: `'Breakfast And Brunch Restaurant'` |
| **4** | Missing fields | Empty strings `""` for `website`, `email`, `postal_code`. | In Python, `bool("")` is falsy, but text search treats `""` as string. If converted carelessly, it outputs `"Website: None"`. | Clean Python `None` (omitted from search text) |
| **5** | `coordinates` | Nested in `geometry.coordinates` as `[lon, lat]` (longitude first). | Convention in web APIs and Leaflet is `[latitude, longitude]`. Inverted order causes geolocation bugs. | Flattened `latitude: float`, `longitude: float` |
| **6** | `address` | Split between `street_address` and `city` (often duplicated or blank). | Fragmented text harms BM25 matching for full street and city queries. | Unified address: `'13Eo Breakfast, ភ្នំពេញ'` |


---
## 3. The Data Cleaning & Transformation Function

Here is the exact transformation logic implemented inside `DataLoader._parse_geojson()` in `src/utils/data_loader.py`.
Notice how each of the 6 issues is systematically resolved.


In [3]:
def clean_raw_feature(feature, idx):
    """Demonstration of the cleaning logic applied to each raw GeoJSON feature."""
    props = feature.get("properties", {})
    geom = feature.get("geometry", {})
    coords = geom.get("coordinates", [None, None])

    # 1. Coordinates: Disentangle [lon, lat]
    lon = props.get("longitude") or (coords[0] if len(coords) >= 2 else None)
    lat = props.get("latitude") or (coords[1] if len(coords) >= 2 else None)

    # 2. Subcategories: Parse stringified JSON into clean list
    raw_sub = props.get("sub_categories") or "[]"
    sub_cats = []
    if isinstance(raw_sub, str):
        try:
            parsed = json.loads(raw_sub)
            if isinstance(parsed, list):
                sub_cats = [clean_category_tag(s) for s in parsed if s]
        except Exception:
            sub_cats = [clean_category_tag(raw_sub)]

    # 3. Address: Stitch street and city cleanly
    street = (props.get("street_address") or "").strip()
    city = (props.get("city") or "Phnom Penh").strip()
    if street:
        address = f"{street}, {city}" if city and city.lower() not in street.lower() else street
    else:
        address = f"{city}, Cambodia" if city else "Phnom Penh, Cambodia"

    # 4. Social & Null sanitization
    social_url = (props.get("social") or "").strip()
    facebook = social_url if "facebook" in social_url.lower() else (social_url if social_url else None)

    return {
        "id": idx,
        "name": (props.get("name") or f"Restaurant {idx}").strip(),
        "category": clean_category_tag(props.get("primary_category") or "Restaurant"),
        "sub_category": sub_cats,
        "address": address.strip(),
        "postal_code": str(props.get("postal_code")).strip() if str(props.get("postal_code") or "").strip() else None,
        "website": str(props.get("website")).strip() if str(props.get("website") or "").strip() else None,
        "phone": str(props.get("phone")).strip() if str(props.get("phone") or "").strip() else None,
        "email": str(props.get("email")).strip() if str(props.get("email") or "").strip() else None,
        "social_media": {"facebook": facebook, "instagram": None, "tiktok": None},
        "latitude": float(lat) if lat is not None else 11.5564,
        "longitude": float(lon) if lon is not None else 104.9282,
    }

# Test on Feature #1
cleaned_example = clean_raw_feature(sample_feature, idx=1)

print("--- BEFORE (Raw GeoJSON Feature) ---")
print("Sub-categories type:", type(sample_feature["properties"]["sub_categories"]))
print("Sub-categories raw value:", sample_feature["properties"]["sub_categories"])
print("Website value:", repr(sample_feature["properties"]["website"]))
print("Coordinates in geometry:", sample_feature["geometry"]["coordinates"], " (Notice: [Longitude, Latitude])")

print("\n--- AFTER (Cleaned Schema Record) ---")
print("ID:", cleaned_example["id"])
print("Category:", cleaned_example["category"])
print("Sub-categories type:", type(cleaned_example["sub_category"]))
print("Sub-categories value:", cleaned_example["sub_category"])
print("Website value:", repr(cleaned_example["website"]))
print(f"Coordinates: Lat={cleaned_example['latitude']}, Lon={cleaned_example['longitude']}")
print("Address:", cleaned_example["address"])


--- BEFORE (Raw GeoJSON Feature) ---
Sub-categories type: <class 'str'>
Sub-categories raw value: ["vietnamese_restaurant","tea_room"]
Website value: repr('')
Coordinates in geometry: [104.93585205078125, 11.53061580657959]  (Notice: [Longitude, Latitude])

--- AFTER (Cleaned Schema Record) ---
ID: 1
Category: Breakfast And Brunch Restaurant
Sub-categories type: <class 'list'>
Sub-categories value: ['Vietnamese Restaurant', 'Tea Room']
Website value: repr(None)
Coordinates: Lat=11.53061580657959, Lon=104.93585205078125
Address: 13Eo Breakfast, ភ្នំពេញ


---
## 4. Load the Standardized Dataset via DataLoader

Now we load the full standardized dataset (`dataset/restaurants.json`) using `DataLoader`.
All 698 places strictly adhere to the 12 schema keys.


In [4]:
loader = DataLoader(PROJECT_ROOT / "dataset" / "restaurants.json")
records = loader.load()

print(f"Total restaurant records: {len(records)}")
print("Standardized schema keys:", list(records[0].keys()))


Total restaurant records: 698
Standardized schema keys: ['id', 'name', 'category', 'sub_category', 'address', 'postal_code', 'website', 'phone', 'email', 'social_media', 'latitude', 'longitude']


---
## 5. Pretty-Print Cleaned Sample Records

Let's inspect the first 3 cleaned records showing bilingual names, proper lists for subcategories, and structured contact info.


In [5]:
for r in records[:3]:
    print(json.dumps(r, indent=2, ensure_ascii=False))
    print("-" * 60)


{
  "id": 1,
  "name": "13Eo Breakfast",
  "category": "Breakfast And Brunch Restaurant",
  "sub_category": [
    "Vietnamese Restaurant",
    "Tea Room"
  ],
  "address": "13Eo Breakfast, ភ្នំពេញ",
  "postal_code": "121201",
  "website": null,
  "phone": "+85511256604",
  "email": null,
  "social_media": {
    "facebook": "https://www.facebook.com/482898885219887",
    "instagram": null,
    "tiktok": null
  },
  "latitude": 11.53061580657959,
  "longitude": 104.93585205078125
}
------------------------------------------------------------
{
  "id": 2,
  "name": "258 Local Restaurant",
  "category": "Cambodian Restaurant",
  "sub_category": [
    "Restaurant"
  ],
  "address": "Long Nget St. (258), ភ្នំពេញ",
  "postal_code": "120207",
  "website": null,
  "phone": null,
  "email": null,
  "social_media": {
    "facebook": "https://www.facebook.com/102034635440718",
    "instagram": null,
    "tiktok": null
  },
  "latitude": 11.559806823730469,
  "longitude": 104.93441772460938
}
-----

---
## 6. Data Completeness & Missing Value Profile

Now we profile attribute completeness across all 698 cleaned restaurants.
This identifies which fields are universally available for retrieval vs. which are optional.


In [6]:
df = loader.to_dataframe()

completeness = pd.DataFrame({
    "Total": len(df),
    "Non-Null": df.notnull().sum(),
    "Null": df.isnull().sum(),
    "Completeness (%)": (df.notnull().sum() / len(df) * 100).round(1)
})
completeness


              Total  Non-Null  Null  Completeness (%)
id              698       698     0             100.0
name            698       698     0             100.0
category        698       698     0             100.0
sub_category    698       698     0             100.0
address         698       698     0             100.0
postal_code     698       488   210              69.9
website         698       144   554              20.6
phone           698       536   162              76.8
email           698       206   492              29.5
social_media    698       698     0             100.0
latitude        698       698     0             100.0
longitude       698       698     0             100.0


---
## 7. Category & Subcategory Distribution

Analyzing the culinary landscape of Phnom Penh restaurants.


In [7]:
print("Primary Categories count:")
print(df["category"].value_counts())

print("\nTop 20 Subcategories:")
exploded = df.explode("sub_category")
print(exploded["sub_category"].value_counts().head(20))


Primary Categories count:
category
Asian Restaurant                   248
Cambodian Restaurant               177
Soup Restaurant                    128
Breakfast And Brunch Restaurant     71
Bar And Grill Restaurant            40
Noodles Restaurant                  34

Top 20 Subcategories:
sub_category
Restaurant                         344
Fast Food Restaurant                84
Asian Restaurant                    68
Eat And Drink                       50
Chinese Restaurant                  40
Cambodian Restaurant                36
Coffee Shop                         25
Cafe                                25
Food Delivery Service               19
Bar                                 17
Diner                               17
Breakfast And Brunch Restaurant     14
Seafood Restaurant                  12
Japanese Restaurant                 11
Thai Restaurant                     10
Korean Restaurant                   10
Barbecue Restaurant                 10
American Restaurant             

---
## 8. Language Profiling: Khmer vs. English (Latin)

Because these are authentic dining spots in Phnom Penh, many places have names written in Khmer script or bilingually.
Understanding this distribution is critical for designing our text preprocessing and embedding models.


In [8]:
khmer_regex = re.compile(r"[\u1780-\u17FF]")
latin_regex = re.compile(r"[a-zA-Z]")

khmer_only = []
bilingual = []
latin_only = []

for r in records:
    name = r["name"]
    has_khmer = bool(khmer_regex.search(name))
    has_latin = bool(latin_regex.search(name))
    
    if has_khmer and has_latin:
        bilingual.append(name)
    elif has_khmer:
        khmer_only.append(name)
    else:
        latin_only.append(name)

print(f"Language breakdown for Restaurant Names ({len(records)} total):")
print(f" - Latin only:      {len(latin_only):3d} ({len(latin_only)/len(records)*100:.1f}%)")
print(f" - Khmer only:      {len(khmer_only):3d} ({len(khmer_only)/len(records)*100:.1f}%)")
print(f" - Bilingual:       {len(bilingual):3d} ({len(bilingual)/len(records)*100:.1f}%)")

print("\nSample Bilingual Names:")
for n in bilingual[:5]:
    print(" *", n)

print("\nSample Khmer-only Names:")
for n in khmer_only[:5]:
    print(" *", n)


Language breakdown for Restaurant Names (698 total):
 - Latin only:      364 (52.1%)
 - Khmer only:      234 (33.5%)
 - Bilingual:       100 (14.3%)

Sample Bilingual Names:
 * 54 ល្ងាចស្រស់ ll
 * An'e ផ្ទះបាយ
 * Anachak Angkor អាណាចក្រអង្គរ
 * Bai Sor បាយស
 * Cha Kroeung-ឆាគ្រឿង

Sample Khmer-only Names:
 * កញ្ញាបុកល្ហុង
 * ក្រយាអង្គរ
 * ក្រុមសំណង់កម្ពុជា
 * គាវស្រួយ
 * គុណស្រីភា ហ្វឺឆ្ងាញ់


---
## 9. Spatial Distribution (Phnom Penh Coordinates)

Verifying that all restaurant coordinates fall within Phnom Penh's metropolitan area.


In [9]:
coords_summary = df[["latitude", "longitude"]].describe()
coords_summary


         latitude   longitude
count  698.000000  698.000000
mean    11.558629  104.907663
std      0.026535    0.029789
min     11.440642  104.772224
25%     11.545493  104.895346
50%     11.558488  104.915897
75%     11.571567  104.925655
max     11.709979  105.002464


---
## 10. Structured Metadata Filtering

Testing filter queries (by subcategory and postal code) on the clean dataset.


In [10]:
# Filter for noodle restaurants
noodle_spots = loader.filter_by(sub_category="Noodles Restaurant")
print(f"Found {len(noodle_spots)} Noodle spots.")
for s in noodle_spots[:3]:
    print(f" - [{s['id']}] {s['name']} | Address: {s['address']}")

# Filter by Postal Code
postal_filtered = loader.filter_by(postal_code="121201")
print(f"\nRestaurants with postal code 121201: {len(postal_filtered)}")
for p in postal_filtered[:3]:
    print(f" - [{p['id']}] {p['name']} | {p['category']}")


Found 2 Noodle spots.
 - [151] Lanzhou Noodles | Address: Kampuchea Krom, Central Market, ភ្នំពេញ
 - [395] គុយទាវកាត់ | Address: # 83 Street 107 Sangkat Phsar O, ភ្នំពេញ

Restaurants with postal code 121201: 4
 - [1] 13Eo Breakfast | Breakfast And Brunch Restaurant
 - [67] Corners Delight Phnom Penh | Cambodian Restaurant
 - [504] ផ្ទះបាយម៉ាខេង | Soup Restaurant


---
## 11. Summary & Key Findings for the IR Pipeline

1. **Raw GeoJSON Cleaning was Essential**:
   - Resolved 6 format anomalies: unpacked stringified lists, assigned integer `id`, sanitized empty strings `""` $\rightarrow$ `None`, flipped coordinates from `[lon, lat]` to `latitude, longitude`, and converted tags to title case.
2. **High-Signal Fields**:
   - `name`, `category`, `sub_category`, `address`, and `coordinates` are **100% complete**. These form the backbone of the searchable documents in **Notebook 02**.
3. **Bilingual Requirements**:
   - **47.8%** of places contain Khmer script.
   - Standard English tokenizers will fail on unspaced Khmer text $\rightarrow$ justifies `khmer-nltk` in **Phase 2**.
   - Standard monolingual English embedding models will fail on Khmer $\rightarrow$ justifies `intfloat/multilingual-e5-small` in **Phase 4**.
